<a href="https://colab.research.google.com/github/jackcase04/scripts/blob/main/NLP_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Jack Case - NLP Assignment 2: Q3

Run the next 3 cells to setup the model that we will use (GloVe)


In [4]:
!pip install gensim

In [5]:
!wget https://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2026-04-19 02:04:14--  https://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-04-19 02:04:14--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip.2’

glove.6B.zip.2      100%[===================>] 822.24M  5.04MB/s    in 2m 39s  

2026-04-19 02:06:54 (5.16 MB/s) - ‘glove.6B.zip.2’ saved [862182613/862182613]

Archive:  glove.6B.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfi

In [6]:
import gensim.downloader as api
model = api.load("glove-wiki-gigaword-100")

For this cell, it will block at the start of execution waiting for upload of the *mikolov_analogy_dataset.txt* file. Once this cell is executing, scroll to the bottom, click *Choose file*, and choose the *mikolov_analogy_dataset.txt* to upload into session storage.

This cell will parse the dataset into a hash map of list's of list's, one for each group, for use in the analogy test later.

In [7]:
from google.colab import files
files.upload()

groups_to_keep = [
    "capital-common-countries",
    "capital-world",
    "currency",
    "city-in-state",
    "family",
    "gram1-adjective-to-adverb",
    "gram2-opposite",
    "gram3-comparative",
    "gram6-nationality-adjective",
    "gram8-plural"
]

with open("mikolov_analogy_dataset.txt") as file:
  current_group = ""
  all_groups = {}

  for line in file:
    start = line[0]

    if start == '/':
      pass
    elif start ==':':
      group = line.split(' ')[1].split('\n')[0]

      if group in groups_to_keep:
        # print(f"Switching to new group: {group}")
        current_group = group

    else:
      if current_group in groups_to_keep:
        analogy = line.strip().split()
        for i in range(len(analogy)):
          analogy[i] = analogy[i].lower()
        # print(f"Analogy: {analogy}")
        if all_groups.get(current_group, 0) == 0:
          all_groups[current_group] = []
        all_groups[current_group].append(analogy)

for group, questions in all_groups.items():
  print(f"{group}: {len(questions)} questions")

Saving mikolov_analogy_dataset.txt to mikolov_analogy_dataset.txt
capital-common-countries: 506 questions
capital-world: 4524 questions
currency: 866 questions
city-in-state: 2467 questions
family: 506 questions
gram1-adjective-to-adverb: 992 questions
gram2-opposite: 812 questions
gram3-comparative: 3510 questions
gram6-nationality-adjective: 3159 questions
gram8-plural: 2202 questions


This is the analogy test cell, with the next one printing the results of this cells calculations.

For every question that occurs in the dataset, we send the first 3 words (a, b, c) to the model to see if it can correctly pick d. We take the top 10 predictions and choose the first one that is not a b or c. Then that is checked against the true value of d to score the models accuracy.

In [8]:
results = {}
total_correct = 0
total_questions = 0

for group, questions in all_groups.items():
    correct = 0

    for a, b, c, d in questions:
        try:
          predictions = model.most_similar(positive=[b, c], negative=[a], topn=10)
        except:
          continue

        predicted_word = ""
        for word, score in predictions:
            if word != a and word != b and word != c:
                predicted_word = word
                break

        if predicted_word == d:
            correct += 1

    total = len(questions)
    results[group] = {
        'correct': correct,
        'total': total
    }
    total_correct += correct
    total_questions += total

print(results)

{'capital-common-countries': {'correct': 475, 'total': 506}, 'capital-world': {'correct': 4024, 'total': 4524}, 'currency': {'correct': 123, 'total': 866}, 'city-in-state': {'correct': 760, 'total': 2467}, 'family': {'correct': 413, 'total': 506}, 'gram1-adjective-to-adverb': {'correct': 242, 'total': 992}, 'gram2-opposite': {'correct': 163, 'total': 812}, 'gram3-comparative': {'correct': 2397, 'total': 3510}, 'gram6-nationality-adjective': {'correct': 2270, 'total': 3159}, 'gram8-plural': {'correct': 1467, 'total': 2202}}


In [9]:
overall_accuracy = total_correct / total_questions
print(f"Overall accuracy: {overall_accuracy:.3f}")
print("NOTE: Accuracy = precision = recall, for this example\n")

print("Groupwise accuracy:")

for group, value in results.items():
  accuracy = value['correct'] / value['total']
  print(f"Group: {group} accuracy = {accuracy}")


Overall accuracy: 0.631
NOTE: Accuracy = precision = recall, for this example

Groupwise accuracy:
Group: capital-common-countries accuracy = 0.9387351778656127
Group: capital-world accuracy = 0.8894783377541998
Group: currency accuracy = 0.1420323325635104
Group: city-in-state accuracy = 0.3080664775030401
Group: family accuracy = 0.8162055335968379
Group: gram1-adjective-to-adverb accuracy = 0.2439516129032258
Group: gram2-opposite accuracy = 0.20073891625615764
Group: gram3-comparative accuracy = 0.6829059829059829
Group: gram6-nationality-adjective accuracy = 0.7185818296929408
Group: gram8-plural accuracy = 0.6662125340599455


Demonstration of how close increase and decrease (opposites) are to each other.

In [10]:
print(model.most_similar("increase", topn=10))
print(model.most_similar("decrease", topn=10))

[('increased', 0.8988391757011414), ('increases', 0.8812485337257385), ('decrease', 0.8812355399131775), ('increasing', 0.8467098474502563), ('reduce', 0.8099488615989685), ('rise', 0.7936252951622009), ('reduced', 0.7800052165985107), ('growth', 0.7789973616600037), ('reduction', 0.776625394821167), ('boost', 0.775901734828949)]
[('increase', 0.881235659122467), ('increases', 0.833343505859375), ('decreases', 0.8315451145172119), ('decreased', 0.8161500692367554), ('increased', 0.8008848428726196), ('decreasing', 0.7891351580619812), ('decline', 0.7761619091033936), ('increasing', 0.7558355927467346), ('rise', 0.7355469465255737), ('proportion', 0.7198822498321533)]


My new types of analogy tests. I chose country-to-language as analogies that were likely to succeed, and programming-language-to-extension as one unlikley to succeed.

In [11]:
all_groups = {
    "country-to-language": [
        ["france", "french", "germany", "german"],
        ["france", "french", "spain", "spanish"],
        ["france", "french", "japan", "japanese"],
    ],
    "programming-language-to-extension": [
        ["python", "py", "java", "jar"],
        ["rust", "rs", "c++", "cpp"],
        ["javascript", "js", "typescript", "ts"]
    ]
}

results = {}
total_correct = 0
total_questions = 0

for group, questions in all_groups.items():
    correct = 0

    for a, b, c, d in questions:
        try:
          predictions = model.most_similar(positive=[b, c], negative=[a], topn=10)
        except:
          continue

        predicted_word = ""
        for word, score in predictions:
            if word != a and word != b and word != c:
                predicted_word = word
                break

        if predicted_word == d:
            correct += 1

    total = len(questions)
    results[group] = {
        'correct': correct,
        'total': total
    }
    total_correct += correct
    total_questions += total

print(results)

{'country-to-language': {'correct': 3, 'total': 3}, 'programming-language-to-extension': {'correct': 0, 'total': 3}}
